# Suspicious Login Attack Detector

This notebook implements a complete suspicious-login detection prototype:

1. Generate **3,000 synthetic rejected-login records** using 10 common email accounts.
2. Create chronological behavior features without future-data leakage.
3. Train a logistic-regression attack classifier.
4. Combine model probability with deterministic security rules.
5. Add a `thret` score from `0` to `1` and recommend an action.
6. Evaluate any new rejected-login attempt by calling one method.

> **Production warning:** The included labels and patterns are synthetic. Retrain and calibrate the model with confirmed real security outcomes before production use.


## 1. Environment

Required packages: `pandas`, `numpy`, `scikit-learn`, `joblib`, and `matplotlib`.

Uncomment the installation command only when the packages are missing.


## 2. Imports, schema, features, and thresholds

In [3]:
from __future__ import annotations

from itertools import groupby

from sklearn.base import BaseEstimator, TransformerMixin

import inspect
import json
import math
import random
from collections import defaultdict, deque
from dataclasses import dataclass
from datetime import datetime, timedelta, timezone
from pathlib import Path
from typing import Any, Iterable

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    precision_recall_fscore_support,
    roc_auc_score,
)
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline


## 3. Synthetic data generation

The generator creates normal user mistakes and four attack scenarios:

- Brute force against one account
- Credential stuffing across multiple accounts
- Distributed attacks against one account
- Possible account takeover from a new device, IP, or location

Documentation-only IP ranges are used, so the generated addresses do not represent real customers.


In [4]:
DEFAULT_SECONDS = 2_592_000  # 30 days
df =pd.read_csv(Path('../data/login_reject_history_3000.csv'))
df['time_to_attempt']=pd.to_datetime(df['time_to_attempt'])


In [5]:



class ApplyTimeChangeTransformer(BaseEstimator, TransformerMixin):

    def __init__(
        self,
        sort_by,
        identity,
        target,
        output_feature,
        condition
    ):
        '''
        sort_by ='time_to_attempt'
        output_feature ='email_attempts_5m'
        identity ='email'
        target ="time_to_attempt"
        condition =lambda seconds: seconds.gt(5 * 60)
        '''
        self.sort_by = sort_by
        self.identity = identity
        self.target = target
        self.output_feature = output_feature
        self.condition =condition


    def fit(self, X, y=None):
        return self

    def transform(self, X):

        # Always use X provided by ColumnTransformer
        tmp_df = X.copy()

        # Validate required columns
        required_columns = {
            self.sort_by,
            self.identity,
            self.target
        }

        missing_columns = required_columns - set(tmp_df.columns)

        if missing_columns:
            raise ValueError(
                f"Missing required columns: {sorted(missing_columns)}"
            )

        # Remember original row positions
        original_position_column = "_original_position"

        tmp_df[original_position_column] = np.arange(
            len(tmp_df)
        )

        # Ensure timestamp is datetime
        tmp_df[self.target] = pd.to_datetime(
            tmp_df[self.target],
            errors="raise"
        )

        # Sort newest to oldest
        tmp_df = tmp_df.sort_values(
            by=self.sort_by,
            ascending=False,
            kind="stable"
        )

        # Calculate difference from next older attempt
        seconds_diff = (
            tmp_df
            .groupby(
                self.identity,
                sort=False
            )[self.target]
            .diff(periods=-1)
            .dt.total_seconds()
        )

        condition_result = self.condition(seconds_diff)

        # More than time limit -> 1
        # Equal/below limit or no older attempt -> 0
        tmp_df[self.output_feature] = np.where(
            condition_result,
            1,
            0
        )

        # Restore original input order
        tmp_df = tmp_df.sort_values(
            by=original_position_column,
            kind="stable"
        )

        # Return exactly one output column
        result = tmp_df[
            [self.output_feature]
        ].copy()

        # Ensure index matches original X
        result.index = X.index

        return result
    def get_feature_names_out(self, input_features=None):
        return np.array([self.output_feature])

# Missing-Value Rules for 22 Engineered Features

| Feature                              | Missing-value rule                                                    |
| ------------------------------------ | --------------------------------------------------------------------- |
| `seconds_since_email_last_attempt`   | Set to `2592000` seconds — 30 days                                    |
| `email_attempts_5m`                  | `0`                                                                   |
| `email_attempts_1h`                  | `0`                                                                   |
| `email_attempts_24h`                 | `0`                                                                   |
| `consecutive_email_failures`         | `0`                                                                   |
| `seconds_since_ip_last_attempt`      | Set to `2592000`                                                      |
| `ip_attempts_5m`                     | `0`                                                                   |
| `ip_attempts_1h`                     | `0`                                                                   |
| `ip_unique_emails_10m`               | `0`                                                                   |
| `ip_unique_emails_1h`                | `0`                                                                   |
| `device_unique_emails_1h`            | `0`                                                                   |
| `email_unique_ips_1h`                | `0`                                                                   |
| `email_unique_ips_24h`               | `0`                                                                   |
| `email_unique_devices_24h`           | `0`                                                                   |
| `email_unique_locations_24h`         | `0`                                                                   |
| `is_new_ip_for_email`                | `0` when the email has no previous history; otherwise calculate `0/1` |
| `is_new_device_for_email`            | `0` when the email has no previous history; otherwise calculate `0/1` |
| `is_new_location_for_email`          | `0` when the email has no previous history; otherwise calculate `0/1` |
| `location_changed_from_last_attempt` | `0` when no previous attempt exists                                   |
| `seconds_since_device_last_attempt`  | Set to `2592000`                                                      |
| `hour_sin`                           | Recalculate from `time_to_attempt`; do not statistically impute       |
| `hour_cos`                           | Recalculate from `time_to_attempt`; do not statistically impute       |




In [6]:
df['_original_sequence']= np.arange(len(df), dtype=int)
df=df.sort_values(['email','time_to_attempt'])

email_group =df.groupby('email',sort=False)
df['email_attempts_5m']=(email_group.rolling(
               window="5min",
               on="time_to_attempt",
               closed="right"
           )["id"]
           .count()
           .astype(int)
           .to_numpy()
)

df['email_attempts_10m']=(email_group.rolling(
               window="10min",
               on="time_to_attempt",
               closed="right"
           )["id"]
           .count()
           .astype(int)
           .to_numpy()
)

df['email_attempts_1h']=(email_group.rolling(
               window="1h",
               on="time_to_attempt",
               closed="right"
           )["id"]
           .count()
           .astype(int)
           .to_numpy()
)


df["seconds_since_email_last_attempt"] = (
    df.sort_values(["email", "time_to_attempt"])
      .groupby("email")["time_to_attempt"]
      .diff()
      .dt.total_seconds()
      .fillna(2592000)
      .astype(int)
)
current_time = pd.Timestamp.now(tz="UTC")

df["seconds_since_email_last_attempt"] = (
    (current_time - df["time_to_attempt"])
    .dt.total_seconds()
    .fillna(2592000)
    .astype(int)
)

df=df.sort_values(['ip','time_to_attempt'])

df['seconds_since_ip_last_attempt']=((current_time - df["time_to_attempt"])
    .dt.total_seconds()
    .fillna(2592000)
    .astype(int))


df['ip_attempts_5m']=(email_group.rolling(
               window="5min",
               on="time_to_attempt",
               closed="right"
           )["id"]
           .count()
           .astype(int)
           .to_numpy()
)

df['ip_attempts_10m']=(email_group.rolling(
               window="10min",
               on="time_to_attempt",
               closed="right"
           )["id"]
           .count()
           .astype(int)
           .to_numpy()
)

hour = df["time_to_attempt"].dt.hour

df["hour_sin"] = np.sin(2 * np.pi * hour / 24)
df["hour_cos"] = np.cos(2 * np.pi * hour / 24)


df["seconds_since_email_last_attempt"] = (
    df.groupby("email")["time_to_attempt"]
      .diff()
      .dt.total_seconds()
      .fillna(DEFAULT_SECONDS)
      .astype("int64")
)

df["seconds_since_ip_last_attempt"] = (
    df.groupby("ip")["time_to_attempt"]
      .diff()
      .dt.total_seconds()
      .fillna(DEFAULT_SECONDS)
      .astype("int64")
)



# df =df.drop(columns=["_original_sequence"])
# df.to_csv("feature_data.csv")


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3000 entries, 1746 to 1024
Data columns (total 19 columns):
 #   Column                            Non-Null Count  Dtype              
---  ------                            --------------  -----              
 0   id                                3000 non-null   int64              
 1   email                             3000 non-null   object             
 2   device_mac_id                     3000 non-null   object             
 3   ip                                3000 non-null   object             
 4   location                          3000 non-null   object             
 5   time_to_attempt                   3000 non-null   datetime64[ns, UTC]
 6   rejection_reason                  3000 non-null   object             
 7   is_suspicious                     3000 non-null   int64              
 8   scenario                          3000 non-null   object             
 9   _original_sequence                3000 non-null   int64          

In [8]:

def add_risk_features(df):
    """
    Add simple login-risk features to the dataframe.

    Returns:
        pandas.DataFrame
    """

    df = df.copy()

    # ---------------------------------------------------------
    # Keep original row order
    # ---------------------------------------------------------
    df["_return_order"] = np.arange(len(df))

    # Sort chronologically for correct historical calculations
    df = df.sort_values("time_to_attempt").reset_index(drop=True)


    # =========================================================
    # 1. device_attempts_5m
    # 2. device_attempts_1h
    # =========================================================

    df["device_attempts_5m"] = (
        df.set_index("time_to_attempt")
          .groupby("device_mac_id")["id"]
          .rolling("5min")
          .count()
          .reset_index(level=0, drop=True)
          .astype(int)
          .values
    )

    df["device_attempts_1h"] = (
        df.set_index("time_to_attempt")
          .groupby("device_mac_id")["id"]
          .rolling("1h")
          .count()
          .reset_index(level=0, drop=True)
          .astype(int)
          .values
    )


    # =========================================================
    # Helper: rolling unique count
    # =========================================================

    def rolling_unique_count(group_col, value_col, window_seconds=3600):

        result = np.zeros(len(df), dtype=int)

        history = defaultdict(deque)
        value_counts = defaultdict(lambda: defaultdict(int))

        for i, row in df.iterrows():

            group = row[group_col]
            value = row[value_col]
            current_time = row["time_to_attempt"]

            queue = history[group]
            counts = value_counts[group]

            cutoff = current_time - pd.Timedelta(seconds=window_seconds)

            # Remove old records
            while queue and queue[0][0] < cutoff:

                old_time, old_value = queue.popleft()

                counts[old_value] -= 1

                if counts[old_value] == 0:
                    del counts[old_value]

            # Add current record
            queue.append((current_time, value))
            counts[value] += 1

            result[i] = len(counts)

        return result


    # =========================================================
    # 3. email_unique_ips_1h
    # =========================================================

    df["email_unique_ips_1h"] = rolling_unique_count(
        group_col="email",
        value_col="ip",
        window_seconds=3600
    )


    # =========================================================
    # 4. email_unique_devices_1h
    # =========================================================

    df["email_unique_devices_1h"] = rolling_unique_count(
        group_col="email",
        value_col="device_mac_id",
        window_seconds=3600
    )


    # =========================================================
    # 5. ip_unique_emails_1h
    # =========================================================

    df["ip_unique_emails_1h"] = rolling_unique_count(
        group_col="ip",
        value_col="email",
        window_seconds=3600
    )


    # =========================================================
    # 6. failed_attempt_streak_email
    # =========================================================

    email_gap = (
        df.groupby("email")["time_to_attempt"]
          .diff()
          .dt.total_seconds()
    )

    email_new_streak = (
        email_gap.isna() |
        (email_gap > 600)
    )

    df["failed_attempt_streak_email"] = (
        df.groupby("email_new_group" if False else "email")
          .cumcount()
    )

    # Calculate actual streak, reset after 10 minutes
    streak_values = []
    streak_tracker = {}

    for _, row in df.iterrows():

        email = row["email"]
        current_time = row["time_to_attempt"]

        if email not in streak_tracker:
            streak = 1

        else:
            previous_time, previous_streak = streak_tracker[email]

            gap = (current_time - previous_time).total_seconds()

            if gap <= 600:
                streak = previous_streak + 1
            else:
                streak = 1

        streak_tracker[email] = (current_time, streak)
        streak_values.append(streak)

    df["failed_attempt_streak_email"] = streak_values


    # =========================================================
    # 7. failed_attempt_streak_ip
    # =========================================================

    streak_values = []
    streak_tracker = {}

    for _, row in df.iterrows():

        ip = row["ip"]
        current_time = row["time_to_attempt"]

        if ip not in streak_tracker:
            streak = 1

        else:
            previous_time, previous_streak = streak_tracker[ip]

            gap = (current_time - previous_time).total_seconds()

            if gap <= 600:
                streak = previous_streak + 1
            else:
                streak = 1

        streak_tracker[ip] = (current_time, streak)
        streak_values.append(streak)

    df["failed_attempt_streak_ip"] = streak_values


    # =========================================================
    # 8. location_change_flag
    # =========================================================

    previous_location = (
        df.groupby("email")["location"]
          .shift(1)
    )

    df["location_change_flag"] = (
        previous_location.notna() &
        (previous_location != df["location"])
    ).astype(int)


    # =========================================================
    # 9. rapid_attempt_flag
    # =========================================================

    # Current feature-engineered columns are used here
    df["rapid_attempt_flag"] = (
        (df["seconds_since_email_last_attempt"].between(0, 30)) |
        (df["seconds_since_ip_last_attempt"].between(0, 30))
    ).astype(int)


    # =========================================================
    # 10. historical_suspicious_rate_email
    # =========================================================

    # IMPORTANT:
    # shift(1) prevents using the current request's target value.

    df["historical_suspicious_rate_email"] = (
        df.groupby("email")["is_suspicious"]
          .transform(
              lambda x: x.shift(1).expanding().mean()
          )
          .fillna(0)
    )


    # =========================================================
    # Restore original dataframe order
    # =========================================================

    df = (
        df.sort_values("_return_order")
          .drop(columns="_return_order")
    )

    return df

In [10]:
final_df= add_risk_features(df)
final_df.to_csv('~/featured_login.csv', index=False)

# Zero Imputer

In [5]:
#  # 
# zero_default_features = [
#    "email_attempts_5m",
#     "email_attempts_1h",
#     "email_attempts_24h",
#     "consecutive_email_failures",
#     "ip_attempts_5m",
#     "ip_attempts_1h",
#     "ip_unique_emails_10m",
#     "ip_unique_emails_1h",
#     "device_unique_emails_1h",
#     "email_unique_ips_1h",
#     "email_unique_ips_24h",
#     "email_unique_devices_24h",
#     "email_unique_locations_24h",
#     "is_new_ip_for_email",
#     "is_new_device_for_email",
#     "is_new_location_for_email",
#     "location_changed_from_last_attempt"
# ]





# df[zero_default_features]=np.nan
# df[['seconds_since_email_last_attempt',
#    'seconds_since_device_last_attempt',
#    'is_new_location_for_email',
#    'is_new_device_for_email',
#    'is_new_ip_for_email',
#    'location_changed_from_last_attempt']]=np.nan 

# preprocessor =ColumnTransformer(
#     transformers=[
#         (
#             'email_attempts_5m',
#             ApplyTimeChangeTransformer(
#                 target='time_to_attempt',
#                 sort_by='time_to_attempt',
#                 output_feature='email_attempts_5m',
#                 identity='email',
#                 condition=lambda second: second.gt(5*60)),
#               ["email", "time_to_attempt"]
#          ),
#         (
#             'email_attempts_1h',
#             ApplyTimeChangeTransformer(
#                 target='time_to_attempt',
#                 sort_by='time_to_attempt',
#                 output_feature='email_attempts_1h',
#                 identity='email',
#                 condition=lambda second: second.gt(1*60*60)),
#               ["email", "time_to_attempt"]
#          ),
#         (
#             'email_attempts_24h',
#             ApplyTimeChangeTransformer(
#                 target='time_to_attempt',
#                 sort_by='time_to_attempt',
#                 output_feature='email_attempts_24h',
#                 identity='email',
#                 condition=lambda second: second.gt(24*60*60)),
#               ["email", "time_to_attempt"]
#          ),
#         (
#             'ip_attempts_5m',
#             ApplyTimeChangeTransformer(
#                 target='time_to_attempt',
#                 sort_by='time_to_attempt',
#                 output_feature='ip_attempts_5m',
#                 identity='ip',
#                 condition=lambda second: second.gt(5*60)),
#               ["ip", "time_to_attempt"]
#          ),
#         (
#             'ip_attempts_1h',
#             ApplyTimeChangeTransformer(
#                 target='time_to_attempt',
#                 sort_by='time_to_attempt',
#                 output_feature='ip_attempts_1h',
#                 identity='ip',
#                 condition=lambda second: second.gt(1*60*60)),
#               ["ip", "time_to_attempt"]
#          ),
#
#         (
#             "keep_email_and_time",
#             "passthrough",
#             ["email", "time_to_attempt",'ip']
#         )
#     ],
#     remainder='passthrough',
#     verbose_feature_names_out=False
# )
#
#
# preprocessor.set_output(transform="pandas")

In [6]:
def add_device_attempts_5m_inplace(
    df: pd.DataFrame,
    *,
    device_col: str = "device_mac_id",
    time_col: str = "time_to_attempt",
    feature_col: str = "device_attempts_5m",
    window_minutes: int = 5,
) -> pd.DataFrame:
    """
    Add the number of previous attempts made by the same device
    during the previous five minutes.

    The current attempt is excluded.

    This function modifies the supplied DataFrame by adding one column.
    It does not reorder the DataFrame.
    """

    # ---------------------------------------------------------
    # Step 1: Validate required columns
    # ---------------------------------------------------------

    required_columns = {
        device_col,
        time_col,
    }

    missing_columns = required_columns.difference(df.columns)

    if missing_columns:
        raise ValueError(
            f"Missing required columns: {sorted(missing_columns)}"
        )

    if df[device_col].isna().any():
        raise ValueError(
            f"Column '{device_col}' contains missing device values."
        )

    # ---------------------------------------------------------
    # Step 2: Convert timestamps
    # ---------------------------------------------------------

    parsed_time = pd.to_datetime(
        df[time_col],
        utc=True,
        errors="raise",
    )

    if parsed_time.isna().any():
        raise ValueError(
            f"Column '{time_col}' contains missing timestamps."
        )

    number_of_rows = len(df)

    if number_of_rows == 0:
        df[feature_col] = pd.Series(dtype="int64")
        return df

    # Convert timestamps to integer nanoseconds.
    #
    # Example:
    # 2026-07-01 10:00:00 becomes one integer value.
    #
    # Integer comparison is cheaper than repeatedly comparing
    # pandas Timestamp objects.
    timestamps_ns = parsed_time.array.asi8

    # ---------------------------------------------------------
    # Step 3: Convert device values to integer codes
    # ---------------------------------------------------------

    # Example:
    #
    # Device-A -> 0
    # Device-B -> 1
    # Device-C -> 2
    #
    # This avoids repeatedly using long device strings as
    # dictionary keys.
    device_codes, unique_devices = pd.factorize(
        df[device_col],
        sort=False,
    )

    # ---------------------------------------------------------
    # Step 4: Sort only row positions
    # ---------------------------------------------------------

    # Suppose original positions are:
    #
    # [0, 1, 2, 3]
    #
    # After sorting by timestamp, the order might become:
    #
    # [2, 0, 3, 1]
    #
    # The DataFrame itself is not sorted or copied.
    chronological_positions = np.argsort(
        timestamps_ns,
        kind="stable",
    )

    # Five minutes expressed in nanoseconds.
    window_ns = pd.Timedelta(
        minutes=window_minutes
    ).value

    # ---------------------------------------------------------
    # Step 5: Prepare state
    # ---------------------------------------------------------

    # device code -> recent timestamps
    #
    # Example:
    #
    # {
    #     0: deque([10:00, 10:02]),
    #     1: deque([10:01])
    # }
    recent_attempts_by_device = defaultdict(deque)

    # The output array follows original DataFrame row positions.
    counts = np.zeros(
        number_of_rows,
        dtype=np.int64,
    )

    # ---------------------------------------------------------
    # Step 6: Process rows chronologically
    # ---------------------------------------------------------

    for row_position in chronological_positions:

        current_time_ns = timestamps_ns[row_position]
        current_device_code = device_codes[row_position]

        cutoff_time_ns = (
            current_time_ns - window_ns
        )

        recent_timestamps = (
            recent_attempts_by_device[current_device_code]
        )

        # Remove timestamps that are more than five minutes old.
        #
        # Each timestamp enters the deque once and leaves once.
        while (
            recent_timestamps
            and recent_timestamps[0] < cutoff_time_ns
        ):
            recent_timestamps.popleft()

        # Everything currently remaining belongs to:
        #
        # [current time - 5 minutes, current time]
        #
        # The current event is not yet inside the deque.
        counts[row_position] = len(recent_timestamps)

        # Add the current event after calculating its feature.
        recent_timestamps.append(current_time_ns)

    # ---------------------------------------------------------
    # Step 7: Add the feature to the original DataFrame
    # ---------------------------------------------------------

    # counts already follows the original row positions.
    # No reverse sorting or merge is necessary.
    df[feature_col] = counts

    return df